# Experimental test 11

              test ('vl',              # llm_model
        'models--kosbu--Llama-3.3-70B-Instruct-AWQ',         # model_ver
        4,                # few_shot_n
        30,                # test_n(# of question for test)
        'Y',              # q_src_yn 
        10,                # iteration num
        'sys_prompt13',   # prompt ver
        5,                # self-consistency number
        0.01,             # temperature
        'ver7'            # excel_verion
        )
### prompt변경하여 테스트


In [1]:
import os
import pandas as pd
from config import config as conf
import re
import numpy as np
from sklearn import metrics



In [2]:
def sc_calc_acc_condition_with_temp_with_sc_model(llm_model, model_ver, few_shot_n, test_n, q_src_yn, ver, p_ver, sc_num, temp, excel_ver):
    tmp = pd.DataFrame()
    df_eval = pd.DataFrame()
    acc_list = []
    path = f'{conf.DATA_PATH}/{conf.ANNO_RESULT}/{model_ver}'
    file_list = os.listdir(path)
    opt_file = [x for x in file_list if x.startswith(f'sc_{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn}_{ver}_{p_ver}_{sc_num}_{temp}_{excel_ver}')]
    opt_file = [x for x in opt_file if x.endswith(f'.csv')] 
    print(f'len of opt_file : {len(opt_file)}')

    df = pd.DataFrame()
    gold_df = pd.DataFrame()

    
    if len(opt_file)>0 : 
        for f in opt_file:
            tmp = pd.read_csv(f'{path}/{f}', index_col =0)
            tmp = tmp.dropna()

            tmp['gold'] = tmp['answer'].apply(lambda x : re.sub(r'[^012]', '', x))
            tmp['o_result'] = tmp['result'].apply(lambda x : re.sub(r'[^012]', '', x))
            tmp = tmp[tmp['o_result'].isin(['1', '0', '2'])]

            
            gold_df = tmp[['id', 'gold']].drop_duplicates()
            chk_cnt = tmp.groupby(['id', 'o_result']).count().reset_index()[['id', 'o_result', 'question']]
            chk_cnt = chk_cnt.rename(columns = {'question': 'cnt'})
            chk_cnt = chk_cnt[chk_cnt['cnt'] == sc_num]
            chk_cnt = chk_cnt.sort_values(by = ['id', 'cnt'], ascending=[True, False]).groupby(['id']).head(1)
            df_eval = pd.merge(gold_df, chk_cnt, on = ['id'])

            print(f'size of the dataset : {df_eval.shape[0]}')
            df_eval['equal_yn'] = np.where(df_eval['gold']==df_eval['o_result'], 1, 0)
            acc = (df_eval['equal_yn'].sum()/df_eval.shape[0])*100  
            acc_list.append(acc)
            df = pd.concat([df, df_eval], axis =0)
            
        df['equal_yn'] = np.where(df['gold']==df['o_result'], 1, 0)
        y_true = df['o_result']
        y_pred = df['gold']
        print(metrics.classification_report(y_true, y_pred, digits=3))

        
        acc = (df['equal_yn'].sum()/df.shape[0])*100            
        print(f'{llm_model}_result_{few_shot_n}_{test_n}_{q_src_yn} : ', acc)
        return acc_list, gold_df


In [3]:
        #   test ('vl',              # llm_model
        # 'models--kosbu--Llama-3.3-70B-Instruct-AWQ',         # model_ver
        # 4,                # few_shot_n
        # 30,                # test_n(# of question for test)
        # 'Y',              # q_src_yn 
        # 10,                # iteration num
        # 'sys_prompt12',   # prompt ver
        # 5,                # self-consistency number
        # 0.01,             # temperature
        # 'ver7'            # excel_verion
        # )

In [4]:
list_, df_ =         sc_calc_acc_condition_with_temp_with_sc_model('vl', 'models--kosbu--Llama-3.3-70B-Instruct-AWQ',  4, 30, 'Y', 10, 'sys_prompt12', 5,  0.01, 'ver7')
print(list_)

len of opt_file : 5
size of the dataset : 19
size of the dataset : 22
size of the dataset : 20
size of the dataset : 24
size of the dataset : 17
              precision    recall  f1-score   support

           0      1.000     0.714     0.833        63
           1      0.533     0.960     0.686        25
           2      0.917     0.786     0.846        14

    accuracy                          0.784       102
   macro avg      0.817     0.820     0.788       102
weighted avg      0.874     0.784     0.799       102

vl_result_4_30_Y :  78.43137254901961
[np.float64(89.47368421052632), np.float64(72.72727272727273), np.float64(75.0), np.float64(75.0), np.float64(82.35294117647058)]


In [7]:
list_, df_ =         sc_calc_acc_condition_with_temp_with_sc_model('vl', 'models--kosbu--Llama-3.3-70B-Instruct-AWQ',  4, 30, 'Y', 10, 'sys_prompt13', 5,  0.01, 'ver7')
print(list_)

len of opt_file : 2
size of the dataset : 19
size of the dataset : 22
              precision    recall  f1-score   support

           0      1.000     0.793     0.885        29
           1      0.500     0.750     0.600         8
           2      0.667     1.000     0.800         4

    accuracy                          0.805        41
   macro avg      0.722     0.848     0.762        41
weighted avg      0.870     0.805     0.821        41

vl_result_4_30_Y :  80.48780487804879
[np.float64(78.94736842105263), np.float64(81.81818181818183)]


In [ ]:
df_